# Worm Moebius

In this notebook, we implement the worm algorithm for the Moebius code for $d = 2 p$ with $p$ odd prime. 

In [ ]:
from jax.typing import ArrayLike
import jax.numpy as jnp
from functools import partial
import numpy as np
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
from coherentinfo.worm import (
    worm_conditional_entropy,
    worm_coherent_information
)
import json
import os
import jax
n_cpus = os.cpu_count()
n_used_cpus = n_cpus
print("Number of CPUs available: {}".format(n_cpus))
print("Number of used CPUs: {}".format(n_used_cpus))
jax.config.update('jax_num_cpu_devices', n_used_cpus)
# jax.config.update("jax_log_compiles", True)
# Devices assumed by JAX
print(jax.devices())

Number of CPUs available: 16
Number of used CPUs: 16
[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7), CpuDevice(id=8), CpuDevice(id=9), CpuDevice(id=10), CpuDevice(id=11), CpuDevice(id=12), CpuDevice(id=13), CpuDevice(id=14), CpuDevice(id=15)]


In [ ]:
moebius_setup = {"length": 3, "width": 3, "p": 3}

worm_setup = {}
worm_setup["num_samples"] = 10 * n_cpus
worm_setup["num_worms"] = 10 * n_cpus
worm_setup["burn_in_steps"] = 3000
worm_setup["max_worm_steps"] = 4000
plaquette_keys_setup = {}
plaquette_keys_setup["worm_master_seed"] = 144
plaquette_keys_setup["error_master_seed"] = 42
gamma_t = 0.2

In [3]:
moebius_code = MoebiusCodeTwoOddPrime(
    length=moebius_setup["length"], width=moebius_setup["width"], d=2 * moebius_setup["p"])
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=2 * moebius_setup["p"], gamma_t=gamma_t
)
print(f"Single error probabilities: {em_lindblad.get_probabilities()}")

Single error probabilities: [0.69740236 0.1367651  0.01363117 0.00180542 0.01363096 0.13676497]


In [4]:
plaquette_conditional_entropy = worm_conditional_entropy(
    gamma_t=gamma_t,
    syndrome_id="plaquette",
    moebius_setup=moebius_setup,
    worm_setup=worm_setup,
    keys_setup=plaquette_keys_setup
)

print(f"Plaquette conditional entropy: {plaquette_conditional_entropy}")

Plaquette conditional entropy: 0.08179366588592529


In [5]:
vertex_keys_setup = {}
vertex_keys_setup["worm_master_seed"] = 144
vertex_keys_setup["error_master_seed"] = 42

vertex_conditional_entropy = worm_conditional_entropy(
    gamma_t=gamma_t,
    syndrome_id="vertex",
    moebius_setup=moebius_setup,
    worm_setup=worm_setup,
    keys_setup=vertex_keys_setup
)

print(f"Vertex conditional entropy: {vertex_conditional_entropy}")

Vertex conditional entropy: 0.4791680872440338


In [ ]:
coherent_information = worm_coherent_information(
    gamma_t=gamma_t,
    moebius_setup=moebius_setup,
    worm_setup=worm_setup,
    plaquette_keys_setup=plaquette_keys_setup,
    vertex_keys_setup=vertex_keys_setup
)

print(f"Coherent Information: {coherent_information}")

Coherent information: 0.4390382468700409


In [ ]:
num_gamma = 20
gamma_t_list = (np.linspace(0.05, 0.3, num_gamma)).tolist()

result = {}
result["gamma_t"] = gamma_t_list

result["plaquette_worm_master_seed"] = (np.arange(0, num_gamma)).tolist()
result["plaquette_error_master_seed"] = (np.arange(0, num_gamma)).tolist()

result["vertex_worm_master_seed"] = (np.arange(0, num_gamma)).tolist()
result["vertex_error_master_seed"] = (np.arange(0, num_gamma)).tolist()

result["coherent_information"] = []

In [ ]:
for index in range(num_gamma):
    plaquette_keys_setup = {}
    plaquette_keys_setup["worm_master_seed"] = result["plaquette_worm_master_seed"][index]
    plaquette_keys_setup["error_master_seed"] = result["plaquette_error_master_seed"][index]

    vertex_keys_setup = {}
    vertex_keys_setup["worm_master_seed"] = result["plaquette_worm_master_seed"][index]
    vertex_keys_setup["error_master_seed"] = result["plaquette_error_master_seed"][index]

    coherent_information = worm_coherent_information(
        gamma_t=result["gamma_t"][index],
        moebius_setup=moebius_setup,
        worm_setup=worm_setup,
        plaquette_keys_setup=plaquette_keys_setup,
        vertex_keys_setup=vertex_keys_setup
    )

    print(f"Gamma: {result["gamma_t"][index]}")
    print(f"Coherent Information: {coherent_information}")

    result["coherent_information"].append(coherent_information)

Gamma: 0.1
Coherent information: 0.9486393332481384
Gamma: 0.15000000000000002
Coherent information: 0.7725446224212646
Gamma: 0.2
Coherent information: 0.36286622285842896
